In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

In [2]:
df = pd.read_parquet("DKHousingPrices.parquet", engine='fastparquet')

In [3]:
df = df.dropna()

In [4]:
# relevant features and target
features = ['year_build', 'no_rooms', 'sqm', 'sqm_price', 'nom_interest_rate%', 'dk_ann_infl_rate%', 'yield_on_mortgage_credit_bonds%']
X = df[features]
y = df['purchase_price']

X = X.fillna(X.mean())

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
param_grid = {
    'n_estimators': [100],
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

In [7]:
# your code here - create the grid search object with the following arguments: cv=5, n_jobs=-1, verbose=3
grid_search = GridSearchCV(estimator=RandomForestRegressor(random_state=42),
                           param_grid=param_grid,
                           cv=5,
                           n_jobs=-1,
                           verbose=3)

In [8]:
# fit the grid search to the training data
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
[CV 1/5] END criterion=gini, max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100;, score=nan total time=   1.3s
[CV 2/5] END criterion=gini, max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100;, score=nan total time=   1.5s
[CV 3/5] END criterion=gini, max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100;, score=nan total time=   1.3s
[CV 3/5] END criterion=gini, max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=nan total time=   1.5s
[CV 2/5] END criterion=gini, max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=nan total time=   1.5s
[CV 1/5] END criterion=gini, max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=nan total time=   1.4s
[CV 4/5] END criterion=gini, max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=nan total time=   1.6s
[CV 5/5] END criterio

KeyboardInterrupt: 

In [ ]:
# get the best estimator from the grid search
best_rf = grid_search.best_estimator_

TypeError: got an unexpected keyword argument 'squared'

In [ ]:
# get the best estimator from the grid search
best_rf = grid_search.best_estimator_

In [ ]:
# extract impurity-based feature importances from `best_rf` and save array as `impurity_importances`
impurity_importances = best_rf.feature_importances_

In [ ]:
# calculate permutation feature importances with `best_rf` and save the mean importances as `permutation_importances`
perm_result = permutation_importance(best_rf, X_test, y_test, n_repeats=30, random_state=42)
permutation_importances = perm_result.importances_mean

In [ ]:
# pair each feature name with its importance and sort them in descending order
features = X_test.columns

impurity_feature_importance = sorted(zip(impurity_importances, features), reverse=True)
sorted_impurity_importances = [value[0] for value in impurity_feature_importance]
sorted_impurity_features = [value[1] for value in impurity_feature_importance]

permutation_indices = np.argsort(permutation_importances)[::-1]
sorted_permutation_importances = permutation_importances[permutation_indices]
sorted_permutation_features = [features[i] for i in permutation_indices]

#cCreate a side-by-side plot for impurity-based and permutation importances
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# plot impurity-based feature importances
axes[0].barh(sorted_impurity_features, sorted_impurity_importances, color='lightcoral')
axes[0].set_title('Impurity-Based Feature Importance')
axes[0].set_xlabel('Importance')
axes[0].invert_yaxis()  # highest importance at the top

# plot permutation feature importances
axes[1].barh(sorted_permutation_features, sorted_permutation_importances, color='steelblue')
axes[1].set_title('Permutation Feature Importance')
axes[1].set_xlabel('Importance')
axes[1].invert_yaxis()  # highest importance at the top

# adjust layout 
plt.tight_layout()
plt.show()

**RANDOM FOREST PREDECTION TEST**